# Variational Autoencoder (VAE) for EEG Data Augmentation

### 1. EEG Data Processing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path 
import mne

from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
def find_project_root():
    p = Path.cwd()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise FileNotFoundError("Project root (with .git) not found")

PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "nm000114"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed_features"
RESULTS_DIR = PROJECT_ROOT / "results"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

In [ ]:
# Helper function to load a single EDF file and return the raw data object

def get_eeg_file(subject_id: str, condition: str):
    return RAW_DATA_DIR / subject_id / "eeg" / f"{subject_id}_task-{condition}_eeg.edf"

In [ ]:
# Test loading one file to ensure everything is working correctly

file_path = get_eeg_file("sub-HS1", "P300")

print(file_path)
print(file_path.exists())

raw = mne.io.read_raw_edf(file_path, preload=True)
print(raw)

In [ ]:
# Scan all files in the raw data directory to ensure we can access them

edf_files = sorted(RAW_DATA_DIR.glob("sub-*/*/*.edf"))

print(f"Number of EDF files found:", len(edf_files))
for file in edf_files[:5]:
    print(file)

In [ ]:
# Parser cell to read all EDF files and extract data and labels for modeling

def infer_label_from_subject(subject_id: str) -> int:
    """
    Infer the label (0 or 1) from the subject ID.
    returns: 
        0 for healthy controls (sub-HS*)
        1 for MDD patients (sub-MDD*)
    """
    subject_upper = subject_id.upper()
    if "HS" in subject_upper:
        return 0
    elif "MDD" in subject_upper:
        return 1 
    else: 
        raise ValueError(f"Could not infer label from subject ID: {subject_id}")
    
def parse_condition_from_filename(filepath: Path) -> str:
    """
    Parse the condition from the filename
    Returns:
        - eyesClosed
        - eyesOpen
        - P300
    """

    name = filepath.name 
    if "task-eyesClosed" in name:
        return "eyesClosed"
    elif "task-eyesOpen" in name:
        return "eyesOpen"
    elif "task-P300" in name:
        return "P300"
    else:
        raise ValueError(f"Could not parse condition from filename: {name}")

In [ ]:
# metadata list to hold all parsed information 

rows = []

for filepath in edf_files:
    subject_id = filepath.parts[-3]  # e.g. subHS1
    condition = parse_condition_from_filename(filepath)
    label = infer_label_from_subject(subject_id)

    rows.append({
        "patient_id": subject_id,
        "recording_id": filepath.stem,
        "label": label,
        "condition": condition,
        "filepath": str(filepath),
    })


# Create a DataFrame from the metadata
metadata_df = pd.DataFrame(rows)

print(metadata_df.head())
print("\nShape:", metadata_df.shape)
print("\nPatients:", metadata_df["patient_id"].nunique())
print("\nClass counts:")
print(metadata_df["label"].value_counts())
print("\nCondition counts:")
print(metadata_df["condition"].value_counts())
print(metadata_df.groupby("patient_id").size().value_counts())

In [ ]:
# save metadata to CSV for future reference
metadata_path = PROJECT_ROOT / "data" / "processed_features" / "metadata.csv"
metadata_df.to_csv(metadata_path, index=False)
print("Saved metadata to:", metadata_path)

In [ ]:
debug_rows = []

for _, row in metadata_df.iterrows():
    raw = mne.io.read_raw_edf(row["filepath"], preload=False, verbose=False)
    debug_rows.append({
        "patient_id": row["patient_id"],
        "condition": row["condition"],
        "filepath": row["filepath"],
        "n_channels": len(raw.ch_names),
        "channel_names": raw.ch_names
    })

debug_df = pd.DataFrame(debug_rows)

print(debug_df["n_channels"].value_counts())

In [ ]:
print("20-channel example:")
print(debug_df.loc[debug_df["n_channels"] == 20, "channel_names"].iloc[0])

print("\n22-channel example:")
print(debug_df.loc[debug_df["n_channels"] == 22, "channel_names"].iloc[0])

In [ ]:
COMMON_CHANNELS = [
'EEG Fp1-LE', 'EEG F3-LE', 'EEG C3-LE', 'EEG P3-LE', 'EEG O1-LE',
 'EEG F7-LE', 'EEG T3-LE', 'EEG T5-LE', 'EEG Fz-LE', 'EEG Fp2-LE', 
 'EEG F4-LE', 'EEG C4-LE', 'EEG P4-LE', 'EEG O2-LE', 'EEG F8-LE', 
 'EEG T4-LE', 'EEG T6-LE', 'EEG Cz-LE', 'EEG Pz-LE', 'EEG A2-A1'
]

print("Number of common channels:", len(COMMON_CHANNELS))


In [ ]:
# Step 1: Extract spectral features (e.g. power in different frequency bands) from each EEG recording

def extract_band_power(raw, COMMON_CHANNELS) -> np.ndarray:
    """
    Extract basic EEG band power features from the raw MNE objects.
    Returns a feature vector.
    """

    data_raw = raw.copy()
    data_raw.pick(COMMON_CHANNELS)
    
    # get data 
    data = data_raw.get_data()      # shape: (n_channels, n_time points)
    sfreq = data_raw.info['sfreq']  # sampling frequency (e.g. 256 Hz)

    # Define frequency bands 
    bands = {
        "delta": (1, 4),     # deep sleep
        "theta": (4, 8),     # drowsiness
        "alpha": (8, 13),    # relaxed 
        "beta":  (13, 30)    # active thinking 
    }

    all_features = []

    # Loop through each channel (ch) and convert signal to frequency domain using Welch's method to estimate power spectral density (psd)
    for ch in data:        
        psd, freqs = mne.time_frequency.psd_array_welch(
            ch,
            sfreq=sfreq,
            fmin=1,
            fmax=30,
            verbose=False
        )

        total_power = psd.sum()
        
        band_features = [
            (
                psd[(freqs >= fmin) & (freqs <= fmax)].mean()
            / total_power                                      # normalize by total power to get relative power in each band
            if total_power > 0 else 0                          # handle case where total power is zero to avoid division by zero
            )                           
            for (fmin, fmax) in bands.values()
            ]
    
        all_features.append(band_features)

    return np.nan_to_num(np.array(all_features).flatten())



In [ ]:
# Build feature matrix 

records = [
    (
        extract_band_power(
            mne.io.read_raw_edf(row["filepath"], preload=True, verbose=False),
            COMMON_CHANNELS
        ),
        row["label"],
        row["patient_id"]
    )
    for _, row in metadata_df.iterrows()
]

X = np.vstack([r[0] for r in records])
y = np.array([r[1] for r in records])
groups = np.array([r[2] for r in records])

print("X.shape:", X.shape)
print("y.shape:", y.shape)
print("Groups shape:", groups.shape)


# VAE for Each Channel

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score

# VAE components 
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

def build_encoder(input_dim, latent_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(128, activation="relu")(inputs)
    x = layers.Dense(64, activation="relu")(x)
    z_mean = layers.Dense(latent_dim)(x)
    z_log_var = layers.Dense(latent_dim)(x)
    return models.Model(inputs, [z_mean, z_log_var])

def build_decoder(output_dim, latent_dim):
    inputs = layers.Input(shape=(latent_dim,))
    x = layers.Dense(64, activation="relu")(inputs)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(output_dim)(x)
    return models.Model(inputs, outputs)

def compute_loss(x, x_recon, z_mean, z_log_var):
    recon = tf.reduce_mean(tf.reduce_sum(tf.square(x - x_recon), axis=1))
    kl = -0.5 * tf.reduce_mean(
        tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)
    )
    return recon + kl

# train VAE
def train_vae(X_data, latent_dim=5, epochs=15):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_data)

    input_dim = X_scaled.shape[1]

    encoder = build_encoder(input_dim, latent_dim)
    decoder = build_decoder(input_dim, latent_dim)

    optimizer = tf.keras.optimizers.Adam()

    dataset = tf.data.Dataset.from_tensor_slices(X_scaled).batch(64)

    for epoch in range(epochs):
        for batch in dataset:
            with tf.GradientTape() as tape:
                z_mean, z_log_var = encoder(batch)
                z = Sampling()([z_mean, z_log_var])
                x_recon = decoder(z)
                loss = compute_loss(batch, x_recon, z_mean, z_log_var)

            grads = tape.gradient(loss, encoder.trainable_weights + decoder.trainable_weights)
            optimizer.apply_gradients(zip(grads, encoder.trainable_weights + decoder.trainable_weights))

    return encoder, decoder, scaler

# generate synthetic data
def generate_samples(decoder, scaler, latent_dim, n_samples):
    z = np.random.normal(size=(n_samples, latent_dim)).astype("float32")
    X_fake_scaled = decoder.predict(z, verbose=0)
    return scaler.inverse_transform(X_fake_scaled)

# Augmentation of training sets
def augment_training_data(X_train, y_train, metadata_train, latent_dim=5, n_fake=100):
    X_aug = [X_train]
    y_aug = [y_train]

    conditions = ["P300", "eyesOpen", "eyesClosed"]

    for cond in conditions:
        for label in [0, 1]:
            mask = (metadata_train["condition"] == cond) & (y_train == label)

            if mask.sum() < 10:
                continue  # skip tiny groups

            X_subset = X_train[mask.values]

            encoder, decoder, scaler = train_vae(X_subset, latent_dim)

            X_fake = generate_samples(decoder, scaler, latent_dim, n_fake)
            y_fake = np.full(n_fake, label)

            X_aug.append(X_fake)
            y_aug.append(y_fake)

    return np.vstack(X_aug), np.hstack(y_aug)

# Apply GroupKFold
gkf = GroupKFold(n_splits=5)

acc_real = []
acc_aug = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    print(f"\nFold {fold+1}")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    metadata_train = metadata_df.iloc[train_idx]

    # Baseline (w/o augmentation)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc_real.append(accuracy_score(y_test, y_pred))

    # VAE Augmentation
    X_aug, y_aug = augment_training_data(X_train, y_train, metadata_train)

    clf_aug = LogisticRegression(max_iter=1000)
    clf_aug.fit(X_aug, y_aug)
    y_pred_aug = clf_aug.predict(X_test)
    acc_aug.append(accuracy_score(y_test, y_pred_aug))

    print(f"Real Acc: {acc_real[-1]:.3f} | Aug Acc: {acc_aug[-1]:.3f}")

# Final Results
print("\n=== FINAL RESULTS ===")
print("Baseline Accuracy:", np.mean(acc_real))
print("Augmented Accuracy:", np.mean(acc_aug))